# Analisis integrado de GoEmotions

In [5]:
from __future__ import annotations

import csv
import json
import math
import statistics
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier

EMOTION_NAMES = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval", "disgust",
    "embarrassment", "excitement", "fear", "gratitude", "grief", "joy", "love",
    "nervousness", "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral",
]

def pretty_json(value: object) -> str:
    return json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True)

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if (repo_root / 'entrega_1').exists() is False:
    # fallback when executed from another cwd
    repo_root = Path('/Users/juan/Documents/proyecto_despliegue_miad')

data_root = repo_root / 'data' / 'go_emotions'
artifacts_root = repo_root / 'entrega_1' / 'artifacts'
artifacts_root.mkdir(parents=True, exist_ok=True)

print('repo_root:', repo_root)
print('data_root exists:', data_root.exists())

repo_root: /Users/juan/Documents/proyecto_despliegue_miad
data_root exists: True


## P02 - Habilitacion y caracterizacion de datos

In [6]:
def summarize_parquet(path: Path) -> dict:
    pf = pq.ParquetFile(path)
    schema = [{"name": f.name, "type": str(f.type)} for f in pf.schema_arrow]
    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "rows": pf.metadata.num_rows,
        "columns": len(schema),
        "schema": schema,
    }


def analyze_simplified_labels(split_path: Path) -> tuple[Counter, Counter]:
    table = pq.read_table(split_path, columns=["labels"])
    labels_array = table["labels"]

    label_counts: Counter = Counter()
    labels_per_sample: Counter = Counter()

    for row in labels_array.to_pylist():
        row_labels = row or []
        labels_per_sample[len(row_labels)] += 1
        for label_id in row_labels:
            if 0 <= label_id < len(EMOTION_NAMES):
                label_counts[EMOTION_NAMES[label_id]] += 1
            else:
                label_counts[f"unknown_{label_id}"] += 1

    return label_counts, labels_per_sample


def count_text_quality(split_path: Path) -> dict:
    table = pq.read_table(split_path, columns=["text"])
    texts = table["text"].to_pylist()
    null_count = sum(1 for t in texts if t is None)
    empty_count = sum(1 for t in texts if isinstance(t, str) and not t.strip())
    return {"null_text_count": null_count, "empty_or_whitespace_text_count": empty_count}


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


p02_dir = artifacts_root / 'p02'
p02_dir.mkdir(parents=True, exist_ok=True)

raw_train = data_root / 'raw' / 'train-00000-of-00001.parquet'
simp_train = data_root / 'simplified' / 'train-00000-of-00001.parquet'
simp_val = data_root / 'simplified' / 'validation-00000-of-00001.parquet'
simp_test = data_root / 'simplified' / 'test-00000-of-00001.parquet'

datasets = {
    'raw_train': raw_train,
    'simplified_train': simp_train,
    'simplified_validation': simp_val,
    'simplified_test': simp_test,
}

profile = {'datasets': {}, 'quality': {}, 'derived': {}}
for name, path in datasets.items():
    profile['datasets'][name] = summarize_parquet(path)
    profile['quality'][name] = count_text_quality(path)

total_label_counts: Counter = Counter()
total_labels_per_sample: Counter = Counter()
split_label_summary = {}
split_labels_per_sample_summary = {}

for split_name, split_path in [('train', simp_train), ('validation', simp_val), ('test', simp_test)]:
    split_label_counts, split_labels_per_sample = analyze_simplified_labels(split_path)
    split_label_summary[split_name] = dict(split_label_counts)
    split_labels_per_sample_summary[split_name] = dict(split_labels_per_sample)
    total_label_counts.update(split_label_counts)
    total_labels_per_sample.update(split_labels_per_sample)

total_simplified_rows = (
    profile['datasets']['simplified_train']['rows']
    + profile['datasets']['simplified_validation']['rows']
    + profile['datasets']['simplified_test']['rows']
)
total_assigned_labels = sum(total_label_counts.values())

profile['derived'] = {
    'total_simplified_rows': total_simplified_rows,
    'total_assigned_labels': total_assigned_labels,
    'avg_labels_per_sample': round(total_assigned_labels / total_simplified_rows, 6),
    'label_counts_by_split': split_label_summary,
    'labels_per_sample_by_split': split_labels_per_sample_summary,
    'label_counts_total': dict(total_label_counts),
    'labels_per_sample_total': dict(total_labels_per_sample),
}

(p02_dir / 'p02_dataset_profile.json').write_text(
    json.dumps(profile, indent=2, ensure_ascii=False), encoding='utf-8'
)

emotion_rows = []
for emotion in EMOTION_NAMES:
    count = total_label_counts.get(emotion, 0)
    pct = (count / total_assigned_labels * 100) if total_assigned_labels else 0
    emotion_rows.append({'emotion': emotion, 'count': count, 'percentage_of_assigned_labels': round(pct, 4)})
emotion_rows.sort(key=lambda r: r['count'], reverse=True)
write_csv(p02_dir / 'p02_emotion_distribution.csv', emotion_rows, ['emotion', 'count', 'percentage_of_assigned_labels'])

lps_rows = []
for k in sorted(total_labels_per_sample):
    c = total_labels_per_sample[k]
    pct = (c / total_simplified_rows * 100) if total_simplified_rows else 0
    lps_rows.append({'labels_per_sample': k, 'count': c, 'percentage_of_samples': round(pct, 4)})
write_csv(p02_dir / 'p02_labels_per_sample_distribution.csv', lps_rows, ['labels_per_sample', 'count', 'percentage_of_samples'])

print('P02 artifacts generated in:', p02_dir)
print('Total rows:', total_simplified_rows)
print('Avg labels/sample:', profile['derived']['avg_labels_per_sample'])
print('Top 5 labels:\n' + pretty_json(emotion_rows[:5]))

P02 artifacts generated in: /Users/juan/Documents/proyecto_despliegue_miad/entrega_1/artifacts/p02
Total rows: 54263
Avg labels/sample: 1.175976
Top 5 labels:
[
  {
    "count": 17772,
    "emotion": "neutral",
    "percentage_of_assigned_labels": 27.8506
  },
  {
    "count": 5122,
    "emotion": "admiration",
    "percentage_of_assigned_labels": 8.0267
  },
  {
    "count": 3687,
    "emotion": "approval",
    "percentage_of_assigned_labels": 5.7779
  },
  {
    "count": 3372,
    "emotion": "gratitude",
    "percentage_of_assigned_labels": 5.2843
  },
  {
    "count": 3093,
    "emotion": "annoyance",
    "percentage_of_assigned_labels": 4.8471
  }
]


## P04 - Exploracion descriptiva (EDA)

In [7]:
def percentile(values: list[int], q: float) -> float:
    if not values:
        return float('nan')
    if q <= 0:
        return float(min(values))
    if q >= 100:
        return float(max(values))
    sorted_vals = sorted(values)
    k = (len(sorted_vals) - 1) * (q / 100)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return float(sorted_vals[int(k)])
    d0 = sorted_vals[f] * (c - k)
    d1 = sorted_vals[c] * (k - f)
    return float(d0 + d1)


def text_stats(texts: list[str]) -> dict:
    char_lens = [len(t) for t in texts]
    word_lens = [len(t.split()) for t in texts]
    return {
        'char_len_mean': round(statistics.fmean(char_lens), 4),
        'char_len_median': statistics.median(char_lens),
        'char_len_p95': round(percentile(char_lens, 95), 4),
        'char_len_min': min(char_lens),
        'char_len_max': max(char_lens),
        'word_len_mean': round(statistics.fmean(word_lens), 4),
        'word_len_median': statistics.median(word_lens),
        'word_len_p95': round(percentile(word_lens, 95), 4),
        'word_len_min': min(word_lens),
        'word_len_max': max(word_lens),
    }


def read_split(split_path: Path) -> tuple[list[str], list[list[int]], list[str]]:
    table = pq.read_table(split_path, columns=['text', 'labels', 'id'])
    texts = table['text'].to_pylist()
    labels = [l or [] for l in table['labels'].to_pylist()]
    ids = table['id'].to_pylist()
    return texts, labels, ids


def summarize_split(texts: list[str], labels: list[list[int]], ids: list[str]) -> tuple[dict, Counter]:
    label_counts = Counter()
    labels_per_sample = [len(l) for l in labels]
    for row in labels:
        for idx in row:
            if 0 <= idx < len(EMOTION_NAMES):
                label_counts[EMOTION_NAMES[idx]] += 1
            else:
                label_counts[f'unknown_{idx}'] += 1

    summary = {
        'rows': len(texts),
        'unique_ids': len(set(ids)),
        'duplicate_ids': len(ids) - len(set(ids)),
        'avg_labels_per_sample': round(statistics.fmean(labels_per_sample), 6),
        'median_labels_per_sample': statistics.median(labels_per_sample),
        'max_labels_per_sample': max(labels_per_sample),
        'text_stats': text_stats(texts),
    }
    return summary, label_counts


p04_dir = artifacts_root / 'p04'
p04_dir.mkdir(parents=True, exist_ok=True)

split_paths = {
    'train': data_root / 'simplified' / 'train-00000-of-00001.parquet',
    'validation': data_root / 'simplified' / 'validation-00000-of-00001.parquet',
    'test': data_root / 'simplified' / 'test-00000-of-00001.parquet',
}

split_data = {}
split_summaries = {}
split_label_counts = {}
for split, path in split_paths.items():
    texts, labels, ids = read_split(path)
    split_data[split] = {'texts': texts, 'labels': labels, 'ids': ids}
    summary, label_counts = summarize_split(texts, labels, ids)
    split_summaries[split] = summary
    split_label_counts[split] = label_counts

train_ids = set(split_data['train']['ids'])
val_ids = set(split_data['validation']['ids'])
test_ids = set(split_data['test']['ids'])

overlap = {
    'train_validation_overlap': len(train_ids & val_ids),
    'train_test_overlap': len(train_ids & test_ids),
    'validation_test_overlap': len(val_ids & test_ids),
}

total_label_counts = Counter()
for c in split_label_counts.values():
    total_label_counts.update(c)
total_assigned_labels = sum(total_label_counts.values())

pair_counts = Counter()
for split in split_data.values():
    for row in split['labels']:
        uniq = sorted(set(row))
        for i, j in combinations(uniq, 2):
            pair_counts[(EMOTION_NAMES[i], EMOTION_NAMES[j])] += 1

top_pairs = [
    {'emotion_a': a, 'emotion_b': b, 'cooccurrence_count': c}
    for (a, b), c in pair_counts.most_common(30)
]

emotion_rows = []
for emotion in EMOTION_NAMES:
    row = {'emotion': emotion}
    total = 0
    for split in ['train', 'validation', 'test']:
        count = split_label_counts[split].get(emotion, 0)
        row[f'{split}_count'] = count
        total += count
    row['total_count'] = total
    row['total_percentage_of_assigned_labels'] = round((total / total_assigned_labels) * 100, 4) if total_assigned_labels else 0.0
    emotion_rows.append(row)
emotion_rows.sort(key=lambda r: r['total_count'], reverse=True)
write_csv(
    p04_dir / 'p04_emotion_counts_by_split.csv',
    emotion_rows,
    ['emotion', 'train_count', 'validation_count', 'test_count', 'total_count', 'total_percentage_of_assigned_labels'],
)

text_stat_rows = []
for split in ['train', 'validation', 'test']:
    s = split_summaries[split]
    ts = s['text_stats']
    text_stat_rows.append({
        'split': split,
        'rows': s['rows'],
        'avg_labels_per_sample': s['avg_labels_per_sample'],
        'char_len_mean': ts['char_len_mean'],
        'char_len_median': ts['char_len_median'],
        'char_len_p95': ts['char_len_p95'],
        'word_len_mean': ts['word_len_mean'],
        'word_len_median': ts['word_len_median'],
        'word_len_p95': ts['word_len_p95'],
        'duplicate_ids': s['duplicate_ids'],
    })
write_csv(
    p04_dir / 'p04_text_length_stats.csv',
    text_stat_rows,
    ['split', 'rows', 'avg_labels_per_sample', 'char_len_mean', 'char_len_median', 'char_len_p95', 'word_len_mean', 'word_len_median', 'word_len_p95', 'duplicate_ids'],
)

write_csv(
    p04_dir / 'p04_top_label_cooccurrence_pairs.csv',
    top_pairs,
    ['emotion_a', 'emotion_b', 'cooccurrence_count'],
)

summary = {
    'split_summaries': split_summaries,
    'id_overlap': overlap,
    'total_simplified_rows': sum(split_summaries[s]['rows'] for s in split_summaries),
    'total_assigned_labels': total_assigned_labels,
    'avg_labels_per_sample_global': round(total_assigned_labels / sum(split_summaries[s]['rows'] for s in split_summaries), 6),
    'top_10_emotions_by_count': [{'emotion': e, 'count': c} for e, c in total_label_counts.most_common(10)],
    'top_10_label_cooccurrence_pairs': top_pairs[:10],
}
(p04_dir / 'p04_eda_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')

print('P04 artifacts generated in:', p04_dir)
print('ID overlap:\n' + pretty_json(overlap))
print('Top 5 emotions:\n' + pretty_json(summary['top_10_emotions_by_count'][:5]))

P04 artifacts generated in: /Users/juan/Documents/proyecto_despliegue_miad/entrega_1/artifacts/p04
ID overlap:
{
  "train_test_overlap": 0,
  "train_validation_overlap": 0,
  "validation_test_overlap": 0
}
Top 5 emotions:
[
  {
    "count": 17772,
    "emotion": "neutral"
  },
  {
    "count": 5122,
    "emotion": "admiration"
  },
  {
    "count": 3687,
    "emotion": "approval"
  },
  {
    "count": 3372,
    "emotion": "gratitude"
  },
  {
    "count": 3093,
    "emotion": "annoyance"
  }
]


## P05 - Linea base predictiva multietiqueta

In [8]:
def read_split_text_labels(path: Path) -> tuple[list[str], list[list[int]]]:
    table = pq.read_table(path, columns=['text', 'labels'])
    texts = table['text'].to_pylist()
    labels = [row or [] for row in table['labels'].to_pylist()]
    return texts, labels


def labels_to_multihot(labels: list[list[int]], num_classes: int) -> np.ndarray:
    y = np.zeros((len(labels), num_classes), dtype=np.int8)
    for i, row in enumerate(labels):
        for idx in row:
            if 0 <= idx < num_classes:
                y[i, idx] = 1
    return y


def apply_threshold_with_fallback(proba: np.ndarray, threshold: float) -> np.ndarray:
    y_pred = (proba >= threshold).astype(np.int8)
    empty_mask = y_pred.sum(axis=1) == 0
    if np.any(empty_mask):
        best_idx = np.argmax(proba[empty_mask], axis=1)
        y_pred[empty_mask, best_idx] = 1
    return y_pred


def metric_block(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        'f1_micro': round(f1_score(y_true, y_pred, average='micro', zero_division=0), 6),
        'f1_macro': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 6),
        'f1_weighted': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 6),
        'precision_micro': round(precision_score(y_true, y_pred, average='micro', zero_division=0), 6),
        'recall_micro': round(recall_score(y_true, y_pred, average='micro', zero_division=0), 6),
    }


p05_dir = artifacts_root / 'p05'
p05_dir.mkdir(parents=True, exist_ok=True)

train_texts, train_labels = read_split_text_labels(data_root / 'simplified' / 'train-00000-of-00001.parquet')
val_texts, val_labels = read_split_text_labels(data_root / 'simplified' / 'validation-00000-of-00001.parquet')
test_texts, test_labels = read_split_text_labels(data_root / 'simplified' / 'test-00000-of-00001.parquet')

y_train = labels_to_multihot(train_labels, len(EMOTION_NAMES))
y_val = labels_to_multihot(val_labels, len(EMOTION_NAMES))
y_test = labels_to_multihot(test_labels, len(EMOTION_NAMES))

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=3,
    max_features=50000,
    sublinear_tf=True,
)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

estimator = SGDClassifier(loss='log_loss', alpha=1e-5, max_iter=30, tol=1e-3, random_state=42)
model = OneVsRestClassifier(estimator, n_jobs=-1)
model.fit(X_train, y_train)

val_proba = model.predict_proba(X_val)
test_proba = model.predict_proba(X_test)

threshold_candidates = np.arange(0.10, 0.91, 0.05)
scan_rows = []
best_threshold = None
best_micro_f1 = -1.0
for t in threshold_candidates:
    y_val_pred_tmp = apply_threshold_with_fallback(val_proba, float(t))
    micro_f1 = f1_score(y_val, y_val_pred_tmp, average='micro', zero_division=0)
    scan_rows.append({'threshold': round(float(t), 2), 'validation_f1_micro': round(float(micro_f1), 6)})
    if micro_f1 > best_micro_f1:
        best_micro_f1 = float(micro_f1)
        best_threshold = float(t)

y_val_pred = apply_threshold_with_fallback(val_proba, best_threshold)
y_test_pred = apply_threshold_with_fallback(test_proba, best_threshold)

val_metrics = metric_block(y_val, y_val_pred)
test_metrics = metric_block(y_test, y_test_pred)

per_class_rows = []
for i, emotion in enumerate(EMOTION_NAMES):
    f1 = f1_score(y_test[:, i], y_test_pred[:, i], zero_division=0)
    support = int(y_test[:, i].sum())
    predicted_positive = int(y_test_pred[:, i].sum())
    per_class_rows.append({
        'emotion': emotion,
        'f1_test': round(float(f1), 6),
        'support_test': support,
        'predicted_positive_test': predicted_positive,
    })
per_class_rows.sort(key=lambda r: r['f1_test'], reverse=True)

results = {
    'model': {
        'type': 'OneVsRest(SGDClassifier log_loss) + TF-IDF',
        'vectorizer': {
            'ngram_range': [1, 2],
            'min_df': 3,
            'max_features': 50000,
            'sublinear_tf': True,
        },
        'estimator': {
            'loss': 'log_loss',
            'alpha': 1e-5,
            'max_iter': 30,
            'tol': 1e-3,
            'random_state': 42,
        },
    },
    'data': {
        'train_rows': int(y_train.shape[0]),
        'validation_rows': int(y_val.shape[0]),
        'test_rows': int(y_test.shape[0]),
        'num_classes': len(EMOTION_NAMES),
    },
    'threshold_selection': {
        'selected_threshold': best_threshold,
        'selection_metric': 'validation f1_micro',
        'best_validation_f1_micro': round(best_micro_f1, 6),
    },
    'metrics': {
        'validation': val_metrics,
        'test': test_metrics,
    },
}

(p05_dir / 'p05_baseline_results.json').write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
write_csv(p05_dir / 'p05_threshold_scan.csv', scan_rows, ['threshold', 'validation_f1_micro'])
write_csv(p05_dir / 'p05_per_class_f1_test.csv', per_class_rows, ['emotion', 'f1_test', 'support_test', 'predicted_positive_test'])

print('P05 artifacts generated in:', p05_dir)
print('Selected threshold:', best_threshold)
print('Validation metrics:\n' + pretty_json(val_metrics))
print('Test metrics:\n' + pretty_json(test_metrics))
print('Top 5 classes by F1:\n' + pretty_json(per_class_rows[:5]))
print('Bottom 5 classes by F1:\n' + pretty_json(per_class_rows[-5:]))

P05 artifacts generated in: /Users/juan/Documents/proyecto_despliegue_miad/entrega_1/artifacts/p05
Selected threshold: 0.20000000000000004
Validation metrics:
{
  "f1_macro": 0.40655,
  "f1_micro": 0.551555,
  "f1_weighted": 0.520298,
  "precision_micro": 0.523904,
  "recall_micro": 0.582288
}
Test metrics:
{
  "f1_macro": 0.412434,
  "f1_micro": 0.555169,
  "f1_weighted": 0.52591,
  "precision_micro": 0.522133,
  "recall_micro": 0.592669
}
Top 5 classes by F1:
[
  {
    "emotion": "gratitude",
    "f1_test": 0.913043,
    "predicted_positive_test": 338,
    "support_test": 352
  },
  {
    "emotion": "amusement",
    "f1_test": 0.793358,
    "predicted_positive_test": 278,
    "support_test": 264
  },
  {
    "emotion": "love",
    "f1_test": 0.769231,
    "predicted_positive_test": 256,
    "support_test": 238
  },
  {
    "emotion": "remorse",
    "f1_test": 0.650407,
    "predicted_positive_test": 67,
    "support_test": 56
  },
  {
    "emotion": "admiration",
    "f1_test": 0.638